In [13]:
import pandas as pd

input_file = r"C:\Users\allad\Desktop\job_offer\AGS\AMRESCO_Automation\Anomaly_detection\AB CarVal 2025 Monthly Production.xlsx"

xls = pd.ExcelFile(input_file)

print("\n================ ANALYSIS STARTED ================\n")

for sheet in xls.sheet_names:
    print(f"\n📄 Sheet: {sheet}")
    print("=" * 90)

    # 1️⃣ Read WITHOUT header
    raw = pd.read_excel(xls, sheet_name=sheet, header=None)

    # 2️⃣ Find the real header row (the one that contains 'Site')
    header_row_idx = None
    for i in range(len(raw)):
        if raw.iloc[i].astype(str).str.contains("Site", case=False).any():
            header_row_idx = i
            break

    if header_row_idx is None:
        print("⚠️ Header row not found. Skipping sheet.")
        continue

    # 3️⃣ Set header correctly
    df = raw.iloc[header_row_idx + 1:].copy()
    df.columns = raw.iloc[header_row_idx]

    # Drop fully empty columns
    df = df.loc[:, df.columns.notna()]

    # 4️⃣ Find % columns by POSITION (not name)
    percent_indices = [
        i for i, c in enumerate(df.columns)
        if isinstance(c, str) and c.strip() == "%"
    ]

    if len(percent_indices) < 2:
        print("⚠️ % columns not found after header fix.")
        continue

    # Rename them meaningfully
    cols = list(df.columns)
    cols[percent_indices[0]] = "Actual_vs_Expected_%"
    cols[percent_indices[1]] = "Actual_vs_Forecast_%"
    df.columns = cols

    # 5️⃣ Analyze both % columns
    for col in ["Actual_vs_Expected_%", "Actual_vs_Forecast_%"]:
        print(f"\n➡️ Analyzing column: {col}")
        print("-" * 70)

        series = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace("%", "", regex=False)
            .str.replace("(", "-", regex=False)
            .str.replace(")", "", regex=False)
        )

        series = pd.to_numeric(series, errors="coerce")

        # Convert decimals → percentages if needed
        if series.abs().max(skipna=True) <= 5:
            series = series * 100

        # Underperforming (< -10%)
        under_df = df[series < -10]

        print("\n📉 Underperforming Sites (< -10%):")
        if under_df.empty:
            print("None")
        else:
            print(under_df.to_string(index=False))
            print(f"\n➡️ Count: {len(under_df)}")

        # Extreme values (>100% or <-100%)
        extreme_df = df[(series > 100) | (series < -100)]

        print("\n⚠️ Extreme % Values (>100 or <-100):")
        if extreme_df.empty:
            print("None")
        else:
            print(extreme_df.to_string(index=False))

print("\n================ ANALYSIS COMPLETED ================\n")


================ ANALYSIS STARTED ================


📄 Sheet: Year to Date

➡️ Analyzing column: Actual_vs_Expected_%
----------------------------------------------------------------------

📉 Underperforming Sites (< -10%):
                       Site State   Actual        Expected           Delta Actual_vs_Expected_%   Forecasted        Delta Actual_vs_Forecast_%
MA: Fairhaven - 279 Mill Rd    MA  3151077    3700216.1863    -549139.1863            -0.148407    3694907.6    -543830.6            -0.147184
 Fresno - 2685 Cornelia Ave    CA 17383394    27393303.829   -10009909.829            -0.365414  29263308.45 -11879914.45            -0.405966
     Charlemont - 0 Tea St     MA  3160322  4478290.114095 -1317968.114095            -0.294302    4333809.7   -1173487.7            -0.270775

➡️ Count: 3

⚠️ Extreme % Values (>100 or <-100):
None

➡️ Analyzing column: Actual_vs_Forecast_%
----------------------------------------------------------------------

📉 Underperforming Sites (< -10%)